# Week 3: Imbalanced Classification and LightGBM Modeling

**Day 1 - Confirm the imbalance problem**

Goal for today: load the fused dataset from Week 2, measure exactly how rare machine failures are, install the libraries you'll need for the rest of the week, and see concretely why accuracy is the wrong metric for this problem before writing a single line of modeling code.



In [ ]:
import pandas as pd
import numpy as np

import lightgbm as lgb
import imblearn

print("lightgbm:", lgb.__version__)
print("imbalanced-learn:", imblearn.__version__)

In [ ]:
DATA_PATH = "ai4i_fused_week2.csv"  # your Week 2 output

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Step 1 - Measure exactly how rare failures are

The brief states failures are under 2% of the data - let's verify that on your actual fused dataset rather than assuming.

In [ ]:
target = "Machine failure"

counts = df[target].value_counts()
pct = df[target].value_counts(normalize=True) * 100

print(counts)
print()
print(pct.round(3))

## Step 2 - Why accuracy will lie to you

Before touching LightGBM, let's prove the point with the simplest possible "model": one that always predicts "no failure," no matter what. If this dummy model still scores high on accuracy, you know accuracy can't be trusted to judge anything for the rest of this week.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score

y = df[target]

y_train, y_test = train_test_split(y, test_size=0.2, stratify=y, random_state=42)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_train), 1)), y_train)
y_pred = dummy.predict(np.zeros((len(y_test), 1)))

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall (failures caught): {recall_score(y_test, y_pred):.4f}")
print(f"F1 score: {f1_score(y_test, y_pred):.4f}")

## Takeaway for Day 1

A model that never predicts a failure can score over 95% accuracy while catching **zero** real failures - exactly the opposite of what a predictive maintenance system needs to do. For the rest of this week, judge every model by **recall**, **F1**, and **ROC-AUC / PR-AUC**, never accuracy alone.

## Day 2 - Build the real 5-fold stratified CV pipeline (no SMOTE yet)

Goal: set up `StratifiedKFold`, train a baseline LightGBM with no resampling tricks, and record real recall/F1/ROC-AUC/PR-AUC numbers across the 5 folds. This is the "before" number Day 3's SMOTE pipeline needs to beat.

### Step 0 - Watch out for a built-in leak

In the AI4I dataset, `Machine failure` is literally defined as 1 whenever *any* of the five failure-mode flags (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) is 1. If you leave those columns in as features, your model will "predict" failure by just reading the answer off the failure-mode columns - you'd get near-perfect recall and it would mean nothing. Drop them, along with ID/text columns that aren't real signal.

In [ ]:
target = "Machine failure"
leak_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
drop_cols = ["UDI", "Product ID", "Type", "timestamp", target] + leak_cols

feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"Using {len(feature_cols)} features:")
print(feature_cols)

X = df[feature_cols]
y = df[target]

### Step 1 - Why stratified matters here

With failures under 2% of rows, a plain random K-fold could easily place very few (or even zero) failures into a given fold by chance, making that fold's metrics meaningless. `StratifiedKFold` forces each fold to keep roughly the same failure rate as the full dataset. Quick check before training anything:

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    fold_rate = y.iloc[test_idx].mean() * 100
    print(f"Fold {i}: {len(test_idx)} rows, failure rate {fold_rate:.3f}%")

### Step 2 - Train a plain LightGBM per fold (the baseline)



In [ ]:
import lightgbm as lgb
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score
)

def sanitize_col_names(df):
    # Replace problematic characters with underscores
    new_columns = []
    for col in df.columns:
        col = col.replace('[', '_').replace(']', '_').replace('<', '_').replace('>', '_').replace(' ', '_').replace('=', '_').replace(',', '_')
        new_columns.append(col)
    df.columns = new_columns
    return df

def run_cv(X, y, cv, model_fn):
    rows = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Sanitize column names for LightGBM
        X_train = sanitize_col_names(X_train.copy())
        X_test = sanitize_col_names(X_test.copy())

        model = model_fn()
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]
        pred = model.predict(X_test)

        rows.append({
            "recall": recall_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "pr_auc": average_precision_score(y_test, proba),
        })
    return pd.DataFrame(rows)

baseline_model_fn = lambda: lgb.LGBMClassifier(random_state=42, verbose=-1)

baseline_results = run_cv(X, y, cv, baseline_model_fn)
baseline_results

In [ ]:
summary = baseline_results.agg(["mean", "std"]).T
summary.columns = ["mean", "std"]
print("Baseline LightGBM (no resampling) - 5-fold CV results:")
summary.round(4)

In [ ]:
baseline_results.to_csv("week3_baseline_cv_results.csv", index=False)
print("Saved week3_baseline_cv_results.csv")

## Takeaway for Day 2

You now have a real baseline: a plain LightGBM, evaluated honestly across 5 stratified folds, with leakage columns removed. Whatever recall/F1/PR-AUC numbers you got above are the bar Day 3's SMOTE pipeline has to clear - and clear by a real margin, since SMOTE adds its own noise too.

## Day 3 - Get SMOTE into the loop correctly

Goal: rebuild yesterday's exact pipeline, but chain `SMOTE -> LightGBM` using an `imblearn` Pipeline instead of plain LightGBM. Everything else - same folds, same model settings - stays identical, so any difference in the numbers is genuinely caused by SMOTE, not by a different setup.

### Why an `imblearn` Pipeline and not manual resampling

If you called `SMOTE().fit_resample()` yourself before the CV loop, you'd be back to the leak from the diagram earlier - synthetic points would be generated using information from rows that end up in the test fold. An `imblearn.pipeline.Pipeline` fixes this automatically: during `.fit()` it resamples only the training data passed in for that fold, and during `.predict()` / `.predict_proba()` it skips the resampling step entirely and just calls the model. You get the correct behavior without having to remember to do it yourself.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

def make_smote_pipeline():
    return ImbPipeline([
        ("smote", SMOTE(random_state=42)),
        ("clf", lgb.LGBMClassifier(random_state=42, verbose=-1)),
    ])

# Reuses the same run_cv() and the same cv (StratifiedKFold, random_state=42) from Day 2,
# so the fold splits are identical - this is what makes the comparison fair.
smote_results = run_cv(X, y, cv, make_smote_pipeline)
smote_results

In [ ]:
smote_summary = smote_results.agg(["mean", "std"]).T
smote_summary.columns = ["mean", "std"]
print("SMOTE + LightGBM - 5-fold CV results:")
smote_summary.round(4)

### Step - Compare against yesterday's baseline, side by side

In [ ]:
baseline_results = pd.read_csv("week3_baseline_cv_results.csv")  # reload in case the kernel restarted

comparison = pd.DataFrame({
    "baseline_mean": baseline_results.mean(),
    "smote_mean": smote_results.mean(),
})
comparison["improvement"] = comparison["smote_mean"] - comparison["baseline_mean"]
comparison.round(4)

### Reading the comparison table

SMOTE usually pushes **recall** up (the model catches more real failures) but can pull **precision** down (it also raises more false alarms), because the synthetic minority samples nudge the decision boundary. Look at the whole table together, not one row in isolation:

- Recall up, F1 and PR-AUC also up -> a genuine win, keep it
- Recall up, but precision collapses and F1/PR-AUC actually got worse -> SMOTE is overfitting to synthetic points; worth revisiting on Day 4 (fewer neighbors, or combine with LightGBM's own class weighting instead of relying on SMOTE alone)

Either outcome is a valid, reportable result - the point of this exercise is to measure it honestly, not to force a win.

In [ ]:
smote_results.to_csv("week3_smote_cv_results.csv", index=False)
comparison.to_csv("week3_baseline_vs_smote_comparison.csv")
print("Saved week3_smote_cv_results.csv and week3_baseline_vs_smote_comparison.csv")

## Day 4 - Tune LightGBM for the imbalance

Goal: try LightGBM's own built-in way of handling class imbalance (`scale_pos_weight`) as an alternative to SMOTE, compare all three approaches side by side, then run a light hyperparameter pass on whichever one wins for you and lock in the best combination.

### `scale_pos_weight` vs SMOTE - different mechanisms, not the same trick

SMOTE changes the **data**: it manufactures synthetic minority-class rows so the training set looks more balanced. `scale_pos_weight` changes the **loss function** instead: it tells LightGBM to penalize a mistake on a failure row more heavily than a mistake on a normal row, without inventing any new data at all. (There's also an `is_unbalance` flag that does something similar automatically - don't set both `scale_pos_weight` and `is_unbalance` at once, they conflict.)

Compute `scale_pos_weight` **only from the training fold**, the same care you took with SMOTE - using the full dataset's ratio would leak a tiny bit of test-set class-balance information into training.

In [ ]:
def run_cv_class_weight(X, y, cv):
    rows = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
        scale_pos_weight = neg / pos  # computed from the training fold only

        # Sanitize column names for LightGBM
        X_train = sanitize_col_names(X_train.copy())
        X_test = sanitize_col_names(X_test.copy())

        model = lgb.LGBMClassifier(random_state=42, verbose=-1, scale_pos_weight=scale_pos_weight)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]
        pred = model.predict(X_test)

        rows.append({
            "recall": recall_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "pr_auc": average_precision_score(y_test, proba),
        })
    return pd.DataFrame(rows)

classweight_results = run_cv_class_weight(X, y, cv)
classweight_results


### Three-way comparison

Reload Tuesday's and Wednesday's saved results so this cell works even if your kernel restarted, then line up all three approaches.

In [ ]:
baseline_results = pd.read_csv("week3_baseline_cv_results.csv")
smote_results = pd.read_csv("week3_smote_cv_results.csv")

three_way = pd.DataFrame({
    "baseline": baseline_results.mean(),
    "smote": smote_results.mean(),
    "class_weight": classweight_results.mean(),
}).round(4)
three_way

A quick note before you pick a winner: combining SMOTE *and* a heavy `scale_pos_weight` in the same model is usually a bad idea, not a bonus - SMOTE already balances the classes to roughly 50/50, so adding strong class weighting on top tends to overcorrect and tank precision. Pick **one** mechanism, not both, based on which row in the table above has the best recall/F1/PR-AUC trade-off for your data.

In [16]:
# Set this to whichever approach won in YOUR three_way table above: "smote" or "class_weight"
WINNING_APPROACH = "class_weight"


### Light hyperparameter pass on the winning approach

A small, manual grid - enough to see if tweaking tree complexity and learning rate moves the needle, without burning your whole day on a full search.

In [ ]:
def build_model(params, approach, y_train=None):
    if approach == "smote":
        return ImbPipeline([
            ("smote", SMOTE(random_state=42)),
            ("clf", lgb.LGBMClassifier(random_state=42, verbose=-1, **params)),
        ])
    else:
        neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
        spw = neg / pos
        return lgb.LGBMClassifier(random_state=42, verbose=-1, scale_pos_weight=spw, **params)

def run_cv_tuned(X, y, cv, approach, params):
    rows = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Sanitize column names for LightGBM
        X_train = sanitize_col_names(X_train.copy())
        X_test = sanitize_col_names(X_test.copy())

        model = build_model(params, approach, y_train)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]
        pred = model.predict(X_test)

        rows.append({
            "recall": recall_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "pr_auc": average_precision_score(y_test, proba),
        })
    return pd.DataFrame(rows).mean()

param_grid = [
    {"num_leaves": 31, "learning_rate": 0.10, "n_estimators": 100},
    {"num_leaves": 63, "learning_rate": 0.05, "n_estimators": 200},
    {"num_leaves": 15, "learning_rate": 0.10, "n_estimators": 150},
]

tuning_results = []
for params in param_grid:
    mean_scores = run_cv_tuned(X, y, cv, WINNING_APPROACH, params)
    tuning_results.append({**params, **mean_scores.to_dict()})

tuning_df = pd.DataFrame(tuning_results)
tuning_df.round(4)

In [ ]:
best_row = tuning_df.loc[tuning_df["f1"].idxmax()]
print("Best hyperparameters by F1:")
print(best_row)

tuning_df.to_csv("week3_hyperparameter_tuning_results.csv", index=False)
print("Saved week3_hyperparameter_tuning_results.csv")

## Takeaway for Day 4

You've now compared three honestly-measured approaches to the imbalance (baseline, SMOTE, class-weighting) on identical folds, and run a light hyperparameter sweep on whichever one won for your data.

## Day 5 - Lock in the final model, confusion matrix, and write-up

Goal: train the final Week 3 model using your winning approach and best hyperparameters from Day 4, get an honest confusion matrix, pull together one summary table of everything you tried this week, and save the trained model so Week 4 (threshold tuning) can pick up directly from here.



### Step 1 - An honest confusion matrix

Fitting one model on the *whole* dataset and testing it on the same data would be leakage - it would look great and mean nothing. Instead, sum the confusion matrix from each of the 5 CV folds: every row in your dataset gets used as a test point exactly once, with a model that never saw it during training, so the totals stay honest.

In [ ]:
from sklearn.metrics import confusion_matrix

def run_cv_confusion(X, y, cv, approach, params):
    total_cm = np.zeros((2, 2), dtype=int)
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Sanitize column names for LightGBM, as done in other CV functions
        X_train = sanitize_col_names(X_train.copy())
        X_test = sanitize_col_names(X_test.copy())

        model = build_model(params, WINNING_APPROACH, y_train)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        total_cm += confusion_matrix(y_test, pred, labels=[0, 1])
    return total_cm

# Extract best_params from best_row as a dictionary
best_params = best_row[["num_leaves", "learning_rate", "n_estimators"]].to_dict()
# Convert n_estimators and num_leaves to int, as LightGBM expects integers
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['num_leaves'] = int(best_params['num_leaves'])

final_cm = run_cv_confusion(X, y, cv, WINNING_APPROACH, best_params)
cm_df = pd.DataFrame(final_cm, index=["actual_no_failure", "actual_failure"],
                      columns=["pred_no_failure", "pred_failure"])
print("Aggregated confusion matrix across all 5 folds (0.5 threshold):")

### Reading it in plant terms

- **False negatives** (actual failure, predicted no failure) are the costly ones - a missed maintenance window, potential unplanned downtime
- **False positives** (actual no failure, predicted failure) cost a wasted inspection, but nothing breaks
- This confusion matrix is at the default 0.5 probability threshold. That threshold isn't necessarily the right trade-off for a maintenance team - Week 4 is exactly about sliding it deliberately to favor catching more failures at the cost of more false alarms, or vice versa.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(final_cm, cmap="Blues")
labels = ["No failure", "Failure"]
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, final_cm[i, j], ha="center", va="center",
                 color="white" if final_cm[i, j] > final_cm.max() / 2 else "black")
ax.set_title("Aggregated Confusion Matrix (5-fold CV, 0.5 threshold)")
plt.tight_layout()
plt.savefig("week3_confusion_matrix.png", dpi=150)
plt.show()

### Step 2 - One summary table for the whole week

In [ ]:
def run_cv_tuned_full(X, y, cv, approach, params):
    rows = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Sanitize column names for LightGBM
        X_train = sanitize_col_names(X_train.copy())
        X_test = sanitize_col_names(X_test.copy())

        model = build_model(params, approach, y_train)
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        pred = model.predict(X_test)
        rows.append({
            "recall": recall_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "pr_auc": average_precision_score(y_test, proba),
        })
    return pd.DataFrame(rows)

final_results = run_cv_tuned_full(X, y, cv, WINNING_APPROACH, best_params)
final_results.to_csv("week3_final_model_cv_results.csv", index=False)

baseline_results = pd.read_csv("week3_baseline_cv_results.csv")
smote_results = pd.read_csv("week3_smote_cv_results.csv")

summary_all = pd.DataFrame({
    "baseline": baseline_results.mean(),
    "smote": smote_results.mean(),
    "class_weight": classweight_results.mean(),
    "final_tuned": final_results.mean(),
}).round(4)
summary_all

### Step 3 - Save the final model for Week 4

The CV results above are your honest performance *estimate*. For an actual model artifact to hand off to Week 4 (threshold tuning needs a concrete trained model, not 5 separate fold-models), fit one final model on the entire dataset using the winning approach and best hyperparameters.

In [ ]:
import joblib

final_model = build_model(best_params, WINNING_APPROACH, y)

# Sanitize column names for LightGBM on the full dataset
X = sanitize_col_names(X.copy())

final_model.fit(X, y)

joblib.dump(final_model, "week3_final_model.joblib")
print("Saved week3_final_model.joblib")